# Assignment 3.1 — Neighborhood Feature Group
**Mostafa Zamaniturk**

Build a SageMaker Feature Store **Neighborhood** feature group from `housing.csv` and `housing_gmaps_data_raw.csv`, then query Brooktree, Fisherman's Wharf, and Los Osos.

**Kernel:** Python 3 (Data Science) in SageMaker Studio (or equivalent with `boto3` / `sagemaker`).

## 1. Setup SageMaker Feature Store

In [ ]:
%pip install 'boto3>1.17.21'

In [ ]:
%pip install 'sagemaker<3.0'

In [6]:
%pip install pandas numpy -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
os._exit(0)

In [1]:
import boto3
import sagemaker

print("boto3:", boto3.__version__)
print("sagemaker:", sagemaker.__version__)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


boto3: 1.43.56
sagemaker: 2.257.6


In [2]:
from sagemaker.session import Session
from sagemaker import get_execution_role

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "sagemaker-featurestore-housing-neighborhood"
role = get_execution_role()

print("region:", region)
print("bucket:", default_s3_bucket_name)
print("role:", role)

region: us-east-1
bucket: sagemaker-us-east-1-682123396235
role: arn:aws:iam::682123396235:role/LabRole


## 2. Load and join datasets

Join housing features to Google Maps metadata on `longitude` / `latitude`. Keep rows that have a `neighborhood-political` value (this becomes the Feature Group primary key).

In [4]:
import math
import time
from time import gmtime, strftime, sleep

import numpy as np
import pandas as pd

housing = pd.read_csv("housing.csv")
gmaps = pd.read_csv("housing_gmaps_data_raw.csv")

print("housing:", housing.shape)
display(housing.head())

print("gmaps:", gmaps.shape)
display(gmaps.head())

print("ocean_proximity values:\n", housing["ocean_proximity"].value_counts())

gmaps_keys = gmaps.drop_duplicates(subset=["longitude", "latitude"])
df = housing.merge(gmaps_keys, on=["longitude", "latitude"], how="inner")

# Primary key comes from neighborhood-political
df = df[df["neighborhood-political"].notna()].copy()
df = df[df["neighborhood-political"].astype(str).str.strip() != ""].copy()
df["neighborhood"] = df["neighborhood-political"].astype(str).str.strip()

print("rows with neighborhood:", df.shape[0])
print("unique neighborhoods:", df["neighborhood"].nunique())
df.head()

housing: (20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


gmaps: (12590, 30)


,street_number,route,locality-political,administrative_area_level_2-political,administrative_area_level_1-political,country-political,postal_code,address,longitude,latitude,...,establishment-natural_feature,airport-establishment-point_of_interest,political-sublocality-sublocality_level_1,administrative_area_level_3-political,post_box,establishment-light_rail_station-point_of_interest-transit_station,establishment-point_of_interest,aquarium-establishment-park-point_of_interest-tourist_attraction-zoo,campground-establishment-lodging-park-point_of_interest-rv_park-tourist_attraction,cemetery-establishment-park-point_of_interest
0,3130,Grizzly Peak Boulevard,Berkeley,Alameda County,California,United States,94705.0,"3130 Grizzly Peak Blvd, Berkeley, CA 94705, USA",-122.23,37.88,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005,Tunnel Road,Oakland,Alameda County,California,United States,94611.0,"2005 Tunnel Rd, Oakland, CA 94611, USA",-122.22,37.86,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6886,Chabot Road,Oakland,Alameda County,California,United States,94618.0,"6886 Chabot Rd, Oakland, CA 94618, USA",-122.24,37.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6365,Florio Street,Oakland,Alameda County,California,United States,94618.0,"6365 Florio St, Oakland, CA 94618, USA",-122.25,37.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5407,Bryant Avenue,Oakland,Alameda County,California,United States,94618.0,"5407 Bryant Ave, Oakland, CA 94618, USA",-122.25,37.84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


ocean_proximity values:
 ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64
rows with neighborhood: 9000
unique neighborhoods: 1306


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,...,airport-establishment-point_of_interest,political-sublocality-sublocality_level_1,administrative_area_level_3-political,post_box,establishment-light_rail_station-point_of_interest-transit_station,establishment-point_of_interest,aquarium-establishment-park-point_of_interest-tourist_attraction-zoo,campground-establishment-lodging-park-point_of_interest-rv_park-tourist_attraction,cemetery-establishment-park-point_of_interest,neighborhood
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Merriewood
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Upper Rockridge
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rockridge
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rockridge
5,-122.25,37.85,52.0,919.0,213.0,413.0,193.0,4.0368,269700.0,NEAR BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rockridge


## 3. Feature engineering (Neighborhood Feature Group)

| Feature | Rule |
|---|---|
| `neighborhood` | primary key from `neighborhood-political` |
| `event_time` | Unix time at ingestion |
| `lt_1h_ocean`, `inland`, `island`, `near_bay`, `near_ocean` | one-hot from `ocean_proximity` (mode per neighborhood) |
| `median_house_value` | mean per neighborhood, capped at 500,000 |
| `median_house_age` | mean of `housing_median_age`, discretized into 10-year bins (`0-9`, `10-19`, ...) |
| `total_households` | mean of `households`, rounded **up** to integer |
| `bedrooms_per_household` | `total_bedrooms / households`; impute missing bedrooms with postal-code average |

Feature Store names cannot contain `<` or spaces, so `<1H OCEAN` → `lt_1h_ocean`, `NEAR BAY` → `near_bay`, etc.

In [5]:
# Impute missing total_bedrooms using postal-code average, then global mean as fallback
df["postal_code"] = df["postal_code"].astype(str)
postal_mean_bedrooms = df.groupby("postal_code")["total_bedrooms"].transform("mean")
df["total_bedrooms_imputed"] = df["total_bedrooms"].fillna(postal_mean_bedrooms)
df["total_bedrooms_imputed"] = df["total_bedrooms_imputed"].fillna(df["total_bedrooms"].mean())

df["bedrooms_per_household"] = df["total_bedrooms_imputed"] / df["households"]

# Aggregate to neighborhood level
neighborhood_df = (
    df.groupby("neighborhood", as_index=False)
    .agg(
        median_house_value=("median_house_value", "mean"),
        median_house_age=("housing_median_age", "mean"),
        total_households=("households", "mean"),
        bedrooms_per_household=("bedrooms_per_household", "mean"),
        ocean_proximity=(
            "ocean_proximity",
            lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0],
        ),
    )
)

# Cap median house value at 500,000
neighborhood_df["median_house_value"] = neighborhood_df["median_house_value"].clip(upper=500000.0)

# Discretize average house age into 10-year bins: 0-9, 10-19, ...
def age_bin(age: float) -> str:
    lo = int(age) // 10 * 10
    return f"{lo}-{lo + 9}"


neighborhood_df["median_house_age"] = neighborhood_df["median_house_age"].apply(age_bin)

# Round average households up to an integer
neighborhood_df["total_households"] = np.ceil(neighborhood_df["total_households"]).astype(int)

# One-hot encode ocean_proximity (ensure all expected categories exist)
ocean_categories = ["<1H OCEAN", "INLAND", "ISLAND", "NEAR BAY", "NEAR OCEAN"]
ocean_dummies = pd.get_dummies(neighborhood_df["ocean_proximity"])
for cat in ocean_categories:
    if cat not in ocean_dummies.columns:
        ocean_dummies[cat] = 0
ocean_dummies = ocean_dummies[ocean_categories].astype(int)

ocean_rename = {
    "<1H OCEAN": "lt_1h_ocean",
    "INLAND": "inland",
    "ISLAND": "island",
    "NEAR BAY": "near_bay",
    "NEAR OCEAN": "near_ocean",
}
ocean_dummies = ocean_dummies.rename(columns=ocean_rename)

neighborhood_df = pd.concat([neighborhood_df.drop(columns=["ocean_proximity"]), ocean_dummies], axis=1)

# Event time = ingestion timestamp (seconds since epoch)
current_time_sec = int(round(time.time()))
neighborhood_df["event_time"] = pd.Series(
    [float(current_time_sec)] * len(neighborhood_df), dtype="float64"
)

# Column order for Feature Group
feature_columns = [
    "neighborhood",
    "event_time",
    "lt_1h_ocean",
    "inland",
    "island",
    "near_bay",
    "near_ocean",
    "median_house_value",
    "median_house_age",
    "total_households",
    "bedrooms_per_household",
]
neighborhood_df = neighborhood_df[feature_columns]

print("neighborhood feature rows:", neighborhood_df.shape)
neighborhood_df.head()

neighborhood feature rows: (1306, 11)


,neighborhood,event_time,lt_1h_ocean,inland,island,near_bay,near_ocean,median_house_value,median_house_age,total_households,bedrooms_per_household
0,28 Palms,1.789846e+09,1,0,0,0,0,222200.000000,20-29,923,1.017335
1,Acorn Industrial,1.789846e+09,0,0,0,1,0,81300.000000,50-59,147,1.659864
2,Adams Hill,1.789846e+09,1,0,0,0,0,250733.333333,30-39,494,1.034649
3,Agua Mansa Industrial Corridor,1.789846e+09,0,1,0,0,0,112300.000000,10-19,516,1.102713
4,Al Tahoe,1.789846e+09,0,1,0,0,0,109180.000000,20-29,249,1.641739


In [6]:
# Quick local preview of the three required neighborhoods (before Feature Store query)
preview_names = ["Brooktree", "Fisherman's Wharf", "Los Osos"]
neighborhood_df[neighborhood_df["neighborhood"].isin(preview_names)]

,neighborhood,event_time,lt_1h_ocean,inland,island,near_bay,near_ocean,median_house_value,median_house_age,total_households,bedrooms_per_household
130,Brooktree,1.789846e+09,1,0,0,0,0,257400.0,0-9,1438,0.336642
390,Fisherman's Wharf,1.789846e+09,0,0,0,1,0,500000.0,50-59,250,1.268000
604,Los Osos,1.789846e+09,0,0,0,0,1,221612.5,10-19,612,1.047885


## 4. Create Neighborhood Feature Group

In [7]:
from sagemaker.feature_store.feature_group import FeatureGroup

neighborhood_feature_group_name = "neighborhood-feature-group-" + strftime("%d-%H-%M-%S", gmtime())

neighborhood_feature_group = FeatureGroup(
    name=neighborhood_feature_group_name,
    sagemaker_session=feature_store_session,
)

def cast_object_to_string(data_frame: pd.DataFrame) -> None:
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")


cast_object_to_string(neighborhood_df)

record_identifier_feature_name = "neighborhood"
event_time_feature_name = "event_time"

neighborhood_feature_group.load_feature_definitions(data_frame=neighborhood_df)
print("Feature group name:", neighborhood_feature_group_name)
neighborhood_feature_group.feature_definitions

Feature group name: neighborhood-feature-group-19-19-37-11


[FeatureDefinition(feature_name='neighborhood', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='event_time', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='lt_1h_ocean', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='inland', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='island', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='near_bay', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='near_ocean', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='median_house_value', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(

In [8]:
def wait_for_feature_group_creation_complete(feature_group: FeatureGroup) -> None:
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


neighborhood_feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=role,
    enable_online_store=True,
)

wait_for_feature_group_creation_complete(feature_group=neighborhood_feature_group)
neighborhood_feature_group.describe()

Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
FeatureGroup neighborhood-feature-group-19-19-37-11 successfully created.


{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:682123396235:feature-group/neighborhood-feature-group-19-19-37-11',
 'FeatureGroupName': 'neighborhood-feature-group-19-19-37-11',
 'RecordIdentifierFeatureName': 'neighborhood',
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'neighborhood',
   'FeatureType': 'String'},
  {'FeatureName': 'event_time', 'FeatureType': 'Fractional'},
  {'FeatureName': 'lt_1h_ocean', 'FeatureType': 'Integral'},
  {'FeatureName': 'inland', 'FeatureType': 'Integral'},
  {'FeatureName': 'island', 'FeatureType': 'Integral'},
  {'FeatureName': 'near_bay', 'FeatureType': 'Integral'},
  {'FeatureName': 'near_ocean', 'FeatureType': 'Integral'},
  {'FeatureName': 'median_house_value', 'FeatureType': 'Fractional'},
  {'FeatureName': 'median_house_age', 'FeatureType': 'String'},
  {'FeatureName': 'total_households', 'FeatureType': 'Integral'},
  {'FeatureName': 'bedrooms_per_household', 'FeatureType': 'Fractional'}],
 'CreationTime': dat

## 5. Ingest records into Feature Store

In [9]:
neighborhood_feature_group.ingest(data_frame=neighborhood_df, max_workers=3, wait=True)
print(f"Ingested {len(neighborhood_df)} neighborhood records.")

Ingested 1306 neighborhood records.


## 6. Query Feature Values

Query the online Feature Store for:

1. Brooktree  
2. Fisherman's Wharf  
3. Los Osos

In [10]:
def get_neighborhood_record(neighborhood_name: str) -> dict:
    response = featurestore_runtime.get_record(
        FeatureGroupName=neighborhood_feature_group_name,
        RecordIdentifierValueAsString=neighborhood_name,
    )
    record = {item["FeatureName"]: item["ValueAsString"] for item in response.get("Record", [])}
    return record


query_neighborhoods = ["Brooktree", "Fisherman's Wharf", "Los Osos"]

for name in query_neighborhoods:
    print("=" * 60)
    print(f"Feature Store record: {name}")
    print("=" * 60)
    record = get_neighborhood_record(name)
    if not record:
        print("No record found.")
    else:
        for k, v in record.items():
            print(f"{k}: {v}")
    print()

Feature Store record: Brooktree
neighborhood: Brooktree
event_time: 1789846441.0
lt_1h_ocean: 1
inland: 0
island: 0
near_bay: 0
near_ocean: 0
median_house_value: 257400.0
median_house_age: 0-9
total_households: 1438
bedrooms_per_household: 0.33664180048046527

Feature Store record: Fisherman's Wharf
neighborhood: Fisherman's Wharf
event_time: 1789846441.0
lt_1h_ocean: 0
inland: 0
island: 0
near_bay: 1
near_ocean: 0
median_house_value: 500000.0
median_house_age: 50-59
total_households: 250
bedrooms_per_household: 1.268

Feature Store record: Los Osos
neighborhood: Los Osos
event_time: 1789846441.0
lt_1h_ocean: 0
inland: 0
island: 0
near_bay: 0
near_ocean: 1
median_house_value: 221612.5
median_house_age: 10-19
total_households: 612
bedrooms_per_household: 1.0478845404823531



In [11]:
# Same three neighborhoods in one BatchGetRecord call (useful extra screenshot)
batch_response = featurestore_runtime.batch_get_record(
    Identifiers=[
        {
            "FeatureGroupName": neighborhood_feature_group_name,
            "RecordIdentifiersValueAsString": query_neighborhoods,
        }
    ]
)
batch_response

{'ResponseMetadata': {'RequestId': '81a096de-ed9a-4f96-9bfd-0c8851ba4234',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '81a096de-ed9a-4f96-9bfd-0c8851ba4234',
   'content-type': 'application/json',
   'content-length': '3120',
   'date': 'Sat, 19 Sep 2026 19:42:51 GMT'},
  'RetryAttempts': 0},
 'Records': [{'FeatureGroupName': 'neighborhood-feature-group-19-19-37-11',
   'RecordIdentifierValueAsString': 'Brooktree',
   'Record': [{'FeatureName': 'neighborhood', 'ValueAsString': 'Brooktree'},
    {'FeatureName': 'event_time', 'ValueAsString': '1789846441.0'},
    {'FeatureName': 'lt_1h_ocean', 'ValueAsString': '1'},
    {'FeatureName': 'inland', 'ValueAsString': '0'},
    {'FeatureName': 'island', 'ValueAsString': '0'},
    {'FeatureName': 'near_bay', 'ValueAsString': '0'},
    {'FeatureName': 'near_ocean', 'ValueAsString': '0'},
    {'FeatureName': 'median_house_value', 'ValueAsString': '257400.0'},
    {'FeatureName': 'median_house_age', 'ValueAsString': '0-9'},
   

### Optional: Athena / Offline Store query

Online `get_record` is enough for the graded screenshots. If you also want offline-store SQL, wait a few minutes after ingestion, then run the cell below.

In [16]:
neighborhood_query = neighborhood_feature_group.athena_query()
neighborhood_table = neighborhood_query.table_name
print("Athena table:", neighborhood_table)

query_string = f"""
SELECT *
FROM "{neighborhood_table}"
WHERE neighborhood IN (
    'Brooktree',
    'Fisherman''s Wharf',
    'Los Osos'
)
"""
print("Running:", query_string)

neighborhood_query.run(
    query_string=query_string,
    output_location=f"s3://{default_s3_bucket_name}/{prefix}/query_results/",
)
neighborhood_query.wait()
athena_df = neighborhood_query.as_dataframe()
athena_df

Athena table: neighborhood_feature_group_19_19_37_11_1789846686
Running: 
SELECT *
FROM "neighborhood_feature_group_19_19_37_11_1789846686"
WHERE neighborhood IN (
    'Brooktree',
    'Fisherman''s Wharf',
    'Los Osos'
)



,neighborhood,event_time,lt_1h_ocean,inland,island,near_bay,near_ocean,median_house_value,median_house_age,total_households,bedrooms_per_household,write_time,api_invocation_time,is_deleted
0,Brooktree,1.789846e+09,1,0,0,0,0,257400.0,0-9,1438,0.336642,2026-09-19 19:46:49.011,2026-09-19 19:41:56.000,False
1,Los Osos,1.789846e+09,0,0,0,0,1,221612.5,10-19,612,1.047885,2026-09-19 19:46:49.037,2026-09-19 19:41:57.000,False
2,Fisherman's Wharf,1.789846e+09,0,0,0,1,0,500000.0,50-59,250,1.268000,2026-09-19 19:46:49.022,2026-09-19 19:41:59.000,False


In [17]:
# Same SQL query, one neighborhood at a time
for name in ["Brooktree", "Fisherman's Wharf", "Los Osos"]:
    # Escape single quotes for Athena SQL (Fisherman's Wharf -> Fisherman''s Wharf)
    name_sql = name.replace("'", "''")
    one_query = f"""
    SELECT *
    FROM "{neighborhood_table}"
    WHERE neighborhood = '{name_sql}'
    """
    print("=" * 60)
    print(one_query)
    neighborhood_query.run(
        query_string=one_query,
        output_location=f"s3://{default_s3_bucket_name}/{prefix}/query_results/",
    )
    neighborhood_query.wait()
    display(neighborhood_query.as_dataframe())


    SELECT *
    FROM "neighborhood_feature_group_19_19_37_11_1789846686"
    WHERE neighborhood = 'Brooktree'
    


,neighborhood,event_time,lt_1h_ocean,inland,island,near_bay,near_ocean,median_house_value,median_house_age,total_households,bedrooms_per_household,write_time,api_invocation_time,is_deleted
0,Brooktree,1.789846e+09,1,0,0,0,0,257400.0,0-9,1438,0.336642,2026-09-19 19:46:49.011,2026-09-19 19:41:56.000,False



    SELECT *
    FROM "neighborhood_feature_group_19_19_37_11_1789846686"
    WHERE neighborhood = 'Fisherman''s Wharf'
    


,neighborhood,event_time,lt_1h_ocean,inland,island,near_bay,near_ocean,median_house_value,median_house_age,total_households,bedrooms_per_household,write_time,api_invocation_time,is_deleted
0,Fisherman's Wharf,1.789846e+09,0,0,0,1,0,500000.0,50-59,250,1.268,2026-09-19 19:46:49.022,2026-09-19 19:41:59.000,False



    SELECT *
    FROM "neighborhood_feature_group_19_19_37_11_1789846686"
    WHERE neighborhood = 'Los Osos'
    


,neighborhood,event_time,lt_1h_ocean,inland,island,near_bay,near_ocean,median_house_value,median_house_age,total_households,bedrooms_per_household,write_time,api_invocation_time,is_deleted
0,Los Osos,1.789846e+09,0,0,0,0,1,221612.5,10-19,612,1.047885,2026-09-19 19:46:49.037,2026-09-19 19:41:57.000,False


## 7. Cleanup 

Delete the Feature Group when you are done to avoid ongoing cost.

In [ ]:
# neighborhood_feature_group.delete()
# print("Deleted:", neighborhood_feature_group_name)